# Sensitivity Labels in Azure AI Search

This notebook demonstrates how to use **Microsoft Purview sensitivity labels** with Azure AI Search to enforce **document-level security** at query time. By the end, you'll have:

1. A **Purview-enabled search index** with integrated vectorization (chunking + embeddings)
2. A **data source** configured to extract sensitivity label metadata during indexing
3. A **skillset** with text splitting, embeddings, and index projections that propagate labels to chunks
4. An **indexer** that ingests labeled documents from Azure Blob Storage
5. **Query-time enforcement** — only documents the user is authorized to see are returned
6. **Elevated read** — admin investigation mode that bypasses label filtering with Purview audit logging

## Architecture

```mermaid
flowchart LR
    subgraph Sources["Blob Storage"]
        D1["📄 Public<br/>Announcement"]
        D2["📄 Confidential<br/>Employee Handbook"]
        D3["📄 Highly Confidential<br/>Financial Projections"]
        D4["📄 Confidential<br/>Project Roadmap"]
    end

    Sources -->|"indexer extracts<br/>content + sensitivity labels"| Indexer["Indexer<br/><i>+ Skillset</i><br/>Text Split → Embeddings"]
    Indexer --> Index["Purview-Enabled Index<br/><code>purview_enabled=True</code><br/>Chunks with labels"]

    Index --> Query["Query-Time Enforcement"]
    Query -->|"x-ms-query-source-authorization"| Filter{"Label Policy<br/>Evaluation"}
    Filter -->|authorized| Results["✅ Filtered Results"]
    Filter -->|unauthorized| Blocked["❌ Document excluded"]

    style Sources fill:#f0f9ff,stroke:#0078d4,color:#000
    style Index fill:#fffdf0,stroke:#d4a000,color:#000
    style Filter fill:#fff0f0,stroke:#d40000,color:#000
```

> **Preview**: This feature uses the `2026-05-01-preview` API via the `azure-search-documents` preview SDK (`--pre`). See [Indexer setup](https://learn.microsoft.com/en-us/azure/search/search-indexer-sensitivity-labels) and [Query-time enforcement](https://learn.microsoft.com/en-us/azure/search/search-query-sensitivity-labels) documentation.

# Section 1: Infrastructure Setup

Complete these steps **before** running the code cells in Section 2. This section is a walkthrough — there are no executable code cells here.

## Step 1: Prerequisites

| Requirement | Details |
|---|---|
| **Azure AI Search** | Basic tier or higher, in the same Microsoft Entra tenant as the querying user |
| **Azure Blob Storage** | To hold the labeled sample documents |
| **Azure OpenAI** | An embedding model deployment (e.g., `text-embedding-3-large`) for integrated vectorization |
| **Microsoft Purview** | Sensitivity label policies [configured](https://learn.microsoft.com/en-us/purview/create-sensitivity-labels) and [applied to documents](https://learn.microsoft.com/en-us/purview/sensitivity-labels) |
| **Entra Roles** | Global Administrator or Privileged Role Administrator — needed to grant Purview permissions to the search service |
| **API version** | `2026-05-01-preview` (handled automatically by `azure-search-documents --pre`) |

### Limitations

- The Azure portal does **not** support creating Purview-enabled indexes
- [Autocomplete](https://learn.microsoft.com/en-us/rest/api/searchservice/documents/autocomplete-post) and [Suggest](https://learn.microsoft.com/en-us/rest/api/searchservice/documents/suggest-post) APIs are not supported
- Guest accounts and cross-tenant queries are not supported
- User-assigned managed identities are **not** supported for Purview role assignments
- Custom Web API skill, GenAI Prompt skill, Knowledge store, Indexer enrichment cache, and Debug sessions do not support labeled documents

## Step 2: Create Azure Resources

If you don't already have these resources, create them using the Azure CLI:

```bash
# Set variables
RESOURCE_GROUP="rg-sensitivity-labels"
LOCATION="eastus"
SEARCH_SERVICE_NAME="search-sensitivity-demo"
STORAGE_ACCOUNT_NAME="stsensitivitydemo"
STORAGE_CONTAINER="labeled-docs"

# Create resource group
az group create --name $RESOURCE_GROUP --location $LOCATION

# Create Azure AI Search (Basic tier minimum)
az search service create \
  --name $SEARCH_SERVICE_NAME \
  --resource-group $RESOURCE_GROUP \
  --location $LOCATION \
  --sku basic

# Create storage account
az storage account create \
  --name $STORAGE_ACCOUNT_NAME \
  --resource-group $RESOURCE_GROUP \
  --location $LOCATION \
  --sku Standard_LRS

# Create blob container
az storage container create \
  --name $STORAGE_CONTAINER \
  --account-name $STORAGE_ACCOUNT_NAME
```

> You also need an **Azure OpenAI** resource with a `text-embedding-3-large` deployment (or equivalent). If you already have one from Notebooks 1-2, reuse it.

## Step 3: Enable System-Assigned Managed Identity

The search service must use its **system-assigned managed identity** to authenticate with Microsoft Purview. User-assigned managed identities are **not** supported for this feature.

```bash
az search service update \
  --name $SEARCH_SERVICE_NAME \
  --resource-group $RESOURCE_GROUP \
  --identity SystemAssigned
```

Verify in the Azure portal: **Search service → Settings → Identity → System assigned → Status: On**

## Step 4: Enable RBAC on the Search Service

Enable role-based access control (RBAC) on your search service. Keep **both** RBAC and API keys enabled to avoid disrupting existing operations.

```bash
az search service update \
  --name $SEARCH_SERVICE_NAME \
  --resource-group $RESOURCE_GROUP \
  --auth-options aadOrApiKey
```

> **Important**: When `purview_enabled` is set to `True` on an index, only RBAC authentication is supported for document operations APIs on that index. API key access is limited to index schema retrieval.

## Step 5: Grant Purview Permissions

A **Global Administrator** or **Privileged Role Administrator** must run the following PowerShell script to grant the search service's managed identity the roles needed to extract sensitivity labels.

```powershell
# Install required modules (run once)
Install-Module -Name Az -Scope CurrentUser
Install-Module -Name Microsoft.Entra -AllowClobber
Import-Module Az.Resources
Connect-Entra -Scopes 'Application.ReadWrite.All'

# Replace with your actual values
$resourceIdWithManagedIdentity = "subscriptions/<subscriptionId>/resourceGroups/<resourceGroup>/providers/Microsoft.Search/searchServices/<searchServiceName>"

$managedIdentityObjectId = (Get-AzResource -ResourceId $resourceIdWithManagedIdentity).Identity.PrincipalId

# Grant Content.SuperUser — Microsoft Information Protection Sync Service
$MIPResourceSP = Get-EntraServicePrincipal -Filter "appID eq '870c4f2e-85b6-4d43-bdda-6ed9a579b725'"
New-EntraServicePrincipalAppRoleAssignment -ServicePrincipalId $managedIdentityObjectId -Principal $managedIdentityObjectId -ResourceId $MIPResourceSP.Id -Id "8b2071cd-015a-4025-8052-1c0dba2d3f64"

# Grant UnifiedPolicy.Tenant.Read — Microsoft Rights Management Services
$MRMSResourceSP = Get-EntraServicePrincipal -Filter "appID eq '00000012-0000-0000-c000-000000000000'"
New-EntraServicePrincipalAppRoleAssignment -ServicePrincipalId $managedIdentityObjectId -Principal $managedIdentityObjectId -ResourceId $MRMSResourceSP.Id -Id "7347eb49-7a1a-43c5-8eac-a5cd1d1c7cf0"
```

| App ID | Service | Role |
|---|---|---|
| `870c4f2e-85b6-4d43-bdda-6ed9a579b725` | Microsoft Information Protection Sync Service | `Content.SuperUser` |
| `00000012-0000-0000-c000-000000000000` | Microsoft Rights Management Services | `UnifiedPolicy.Tenant.Read` |

> **Governance**: Engage your internal security or compliance teams to review before proceeding with these role assignments.

## Step 6: Prepare Sample Documents with Sensitivity Labels

This notebook includes 4 sample `.docx` files in the [`data/sensitivity-labels/`](../data/sensitivity-labels/) folder. **You must apply sensitivity labels in Microsoft Word before uploading them to Blob Storage** — labels cannot be applied programmatically from this notebook.

| File | Content | Apply This Label |
|---|---|---|
| `public-announcement.docx` | Company picnic announcement | **General** or **Public** |
| `employee-handbook.docx` | Remote work policy | **Confidential** |
| `financial-projections.docx` | Q3 2025 financial forecasts | **Highly Confidential** |
| `project-roadmap.docx` | IoT product roadmap | **Confidential** |

### Steps

1. **Open each file in Microsoft Word** (desktop or web)
2. **Apply the sensitivity label**: Home → Sensitivity → select the label from the table above
3. **Save** the file
4. **Upload all labeled documents to your blob container**:

```bash
az storage blob upload-batch \
  --source ../data/sensitivity-labels \
  --destination $STORAGE_CONTAINER \
  --account-name $STORAGE_ACCOUNT_NAME \
  --pattern "*.docx" \
  --overwrite
```

> **Note**: If your tenant doesn't have the exact label names above, use the closest equivalents (e.g., "Internal" instead of "Confidential"). The important thing is to have documents with **different** sensitivity levels.

## Step 7: Configure `.env` File

Create a `.env` file in the notebook directory (or add to your existing one) with these variables:

```
AZURE_SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
AZURE_STORAGE_CONNECTION_STRING=DefaultEndpointsProtocol=https;AccountName=<name>;AccountKey=<key>;EndpointSuffix=core.windows.net
AZURE_STORAGE_CONTAINER=labeled-docs
AZURE_OPENAI_ENDPOINT=https://<your-openai-resource>.openai.azure.com
EMBEDDING_DEPLOYMENT=text-embedding-3-large
EMBEDDING_MODEL=text-embedding-3-large
```

## Step 8: Assign RBAC Roles for Querying

Assign these roles to the users who will query the index:

| Role | Purpose | Who |
|---|---|---|
| **Search Index Data Reader** | Standard queries with label enforcement | All querying users/apps |
| **Search Index Data Contributor** | Elevated read (admin investigation) | Admins only |

```bash
# Standard query access
az role assignment create \
  --assignee <user-object-id> \
  --role "Search Index Data Reader" \
  --scope /subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Search/searchServices/<name>

# Elevated read access (admin only)
az role assignment create \
  --assignee <admin-object-id> \
  --role "Search Index Data Contributor" \
  --scope /subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Search/searchServices/<name>
```

---

✅ **Setup complete** — proceed to Section 2 to create and query the Purview-enabled index.

# Section 2: Execution

Run the cells below in order after completing all infrastructure setup steps above.

## Phase 3A: Setup & Configuration

In [ ]:
# Install required packages (uncomment if not already installed)
# %pip install azure-search-documents --pre azure-identity python-dotenv requests

In [ ]:
import os
import json
import subprocess
import time
from datetime import timedelta
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer,
    SearchIndexer,
    FieldMapping,
    IndexingSchedule,
    SearchIndexerSkillset,
    SplitSkill,
    AzureOpenAIEmbeddingSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
)

load_dotenv()

# Azure AI Search
search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]

# Azure Blob Storage
storage_connection_string = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
storage_container = os.environ["AZURE_STORAGE_CONTAINER"]

# Azure OpenAI (for integrated vectorization)
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
embedding_deployment = os.environ["EMBEDDING_DEPLOYMENT"]
embedding_model = os.environ["EMBEDDING_MODEL"]

# Authenticate with DefaultAzureCredential (RBAC required for Purview-enabled indexes)
credential = DefaultAzureCredential()

# SDK clients
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)

print(f"Search endpoint:      {search_endpoint}")
print(f"Storage container:    {storage_container}")
print(f"OpenAI endpoint:      {openai_endpoint}")
print(f"Embedding:            {embedding_deployment} / {embedding_model}")
print(f"Auth method:          DefaultAzureCredential (RBAC)")

In [ ]:
# Resource names used throughout this notebook
INDEX_NAME = "sensitivity-labels-index"
DATA_SOURCE_NAME = "sensitivity-labels-ds"
SKILLSET_NAME = "sensitivity-labels-skillset"
INDEXER_NAME = "sensitivity-labels-indexer"

## Phase 3B: Create Purview-Enabled Index

We create a search index with:
- **`purview_enabled=True`** — enables sensitivity label enforcement (permanent, cannot be changed after creation)
- **`sensitivity_label_id=True`** on the `sensitivityLabel` field — marks it for label-based filtering
- **Vector search** with an Azure OpenAI vectorizer for integrated vectorization
- **Semantic configuration** for L2 reranking
- **Index projections** will map the sensitivity label from parent documents to child chunks

> **Important**: Once `purview_enabled` is set to `True`, only RBAC authentication is supported for document operations on this index.

In [ ]:
# Define the index schema
fields = [
    SimpleField(name="chunk_id", type=SearchFieldDataType.String, key=True, filterable=True),
    SimpleField(name="parent_id", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="title", type=SearchFieldDataType.String, filterable=True, retrievable=True),
    SearchableField(name="content", type=SearchFieldDataType.String, retrievable=True),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=3072,
        vector_search_profile_name="vector-profile",
    ),
    SimpleField(
        name="sensitivityLabel",
        type=SearchFieldDataType.String,
        filterable=True,
        retrievable=True,
        sensitivity_label_id=True,  # Marks this field for label-based access control
    ),
]

# Vector search configuration with Azure OpenAI vectorizer
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-algorithm")],
    profiles=[
        VectorSearchProfile(
            name="vector-profile",
            algorithm_configuration_name="hnsw-algorithm",
            vectorizer_name="openai-vectorizer",
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name="openai-vectorizer",
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=openai_endpoint,
                deployment_name=embedding_deployment,
                model_name=embedding_model,
            ),
        )
    ],
)

# Semantic configuration for reranking
semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
        title_field=SemanticField(field_name="title"),
    ),
)

# Create the Purview-enabled index
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=SemanticSearch(configurations=[semantic_config]),
    purview_enabled=True,  # Permanent — cannot be changed after creation
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created (purview_enabled={result.purview_enabled})")
print(f"Fields: {[f.name for f in result.fields]}")

## Phase 3C: Create Data Source

The data source connection points to the blob container with your labeled documents. The key setting is `indexer_permission_options=["sensitivityLabel"]`, which instructs the indexer to extract sensitivity label metadata during ingestion.

In [ ]:
# Create the data source connection with sensitivity label extraction enabled
data_source = SearchIndexerDataSourceConnection(
    name=DATA_SOURCE_NAME,
    type="azureblob",
    connection_string=storage_connection_string,
    container=SearchIndexerDataContainer(name=storage_container),
    indexer_permission_options=["sensitivityLabel"],  # Extract sensitivity labels during indexing
)

result = indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source '{result.name}' created")
print(f"Type: {result.type}")
print(f"Permission options: {result.indexer_permission_options}")

## Phase 3D: Create Skillset with Index Projections

The skillset handles integrated vectorization:
1. **Text Split skill** — chunks documents into smaller passages
2. **Azure OpenAI Embedding skill** — generates vector embeddings for each chunk

**Index projections** propagate the `sensitivityLabel` from the parent document to every child chunk. Without this mapping, chunk rows won't be filtered correctly at query time.

In [ ]:
# Skills: text splitting + embedding generation
split_skill = SplitSkill(
    name="split-skill",
    description="Split documents into chunks",
    context="/document",
    inputs=[InputFieldMappingEntry(name="text", source="/document/content")],
    outputs=[OutputFieldMappingEntry(name="textItems", target_name="chunks")],
    text_split_mode="pages",
    maximum_page_length=2000,
    page_overlap_length=200,
)

embedding_skill = AzureOpenAIEmbeddingSkill(
    name="embedding-skill",
    description="Generate embeddings for each chunk",
    context="/document/chunks/*",
    inputs=[InputFieldMappingEntry(name="text", source="/document/chunks/*")],
    outputs=[OutputFieldMappingEntry(name="embedding", target_name="content_vector")],
    resource_url=openai_endpoint,
    deployment_name=embedding_deployment,
    model_name=embedding_model,
)

# Index projections: map parent-level fields (including sensitivity label) to each chunk
index_projections = SearchIndexerIndexProjection(
    selectors=[
        SearchIndexerIndexProjectionSelector(
            target_index_name=INDEX_NAME,
            parent_key_field_name="parent_id",
            source_context="/document/chunks/*",
            mappings=[
                InputFieldMappingEntry(name="content", source="/document/chunks/*"),
                InputFieldMappingEntry(name="content_vector", source="/document/chunks/*/content_vector"),
                InputFieldMappingEntry(name="title", source="/document/metadata_storage_name"),
                InputFieldMappingEntry(name="sensitivityLabel", source="/document/metadata_sensitivity_label"),
            ],
        )
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode="skipIndexingParentDocuments"
    ),
)

# Create the skillset
skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    description="Text splitting + embeddings with sensitivity label projection",
    skills=[split_skill, embedding_skill],
    index_projection=index_projections,
)

result = indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{result.name}' created")
print(f"Skills: {[s.name for s in result.skills]}")
print(f"Index projections target: {result.index_projection.selectors[0].target_index_name}")

## Phase 3E: Create & Run Indexer

The indexer connects the data source, skillset, and index. It maps `metadata_sensitivity_label` (extracted by the indexer from Purview) to the `sensitivityLabel` field in the index.

The schedule is set to 5 minutes (the minimum supported interval) so label changes on source documents are picked up quickly.

In [ ]:
# Create the indexer with field mappings and schedule
indexer = SearchIndexer(
    name=INDEXER_NAME,
    data_source_name=DATA_SOURCE_NAME,
    target_index_name=INDEX_NAME,
    skillset_name=SKILLSET_NAME,
    field_mappings=[
        FieldMapping(
            source_field_name="metadata_sensitivity_label",
            target_field_name="sensitivityLabel",
        )
    ],
    schedule=IndexingSchedule(interval=timedelta(minutes=5)),
)

result = indexer_client.create_or_update_indexer(indexer)
print(f"Indexer '{result.name}' created")
print(f"Data source: {result.data_source_name}")
print(f"Target index: {result.target_index_name}")
print(f"Skillset: {result.skillset_name}")

# Trigger an immediate run
indexer_client.run_indexer(INDEXER_NAME)
print(f"\nIndexer '{INDEXER_NAME}' run triggered.")

## Phase 3F: Monitor Indexer Status

Poll the indexer until it completes, then verify that sensitivity labels were extracted and attached to the indexed chunks.

In [ ]:
# Poll indexer status until complete
while True:
    status = indexer_client.get_indexer_status(INDEXER_NAME)
    last_result = status.last_result
    if last_result is not None:
        print(f"Status: {last_result.status} | Items processed: {last_result.items_processed} | Items failed: {last_result.items_failed}")
        if last_result.status in ("success", "transientFailure"):
            break
        if last_result.status == "error":
            print(f"Error: {last_result.errors}")
            break
    else:
        print("Indexer is running...")
    time.sleep(10)

print(f"\nIndexer finished with status: {last_result.status}")

In [ ]:
# Verify sensitivity labels in indexed documents
# Note: This basic search does NOT enforce label policies (no user token provided).
# Documents without labels will still be returned in this verification query.
search_client = SearchClient(endpoint=search_endpoint, index_name=INDEX_NAME, credential=credential)

results = search_client.search(
    search_text="*",
    select=["chunk_id", "title", "sensitivityLabel"],
    top=10,
)

print("Indexed chunks with sensitivity labels:\n")
for doc in results:
    label = doc.get("sensitivityLabel", "None")
    print(f"  [{label:>30}]  {doc['title']}  (chunk: {doc['chunk_id'][:20]}...)")

## Phase 3G: Query with Sensitivity Label Enforcement

Query-time enforcement requires **two tokens**:

| Header | Purpose | How to get |
|---|---|---|
| `Authorization: Bearer <app-token>` | Application RBAC role (Search Index Data Reader) | Handled automatically by `DefaultAzureCredential` via the SDK |
| `x-ms-query-source-authorization: <user-token>` | User identity for label evaluation | `az account get-access-token --resource https://search.azure.com` |

The `x-ms-query-source-authorization` header carries the **user** token (without the `Bearer` prefix). Azure AI Search evaluates the user's Purview permissions and returns only documents whose sensitivity label the user is authorized to access with `EXTRACT` rights.

> **Note**: The `az account get-access-token` approach is for **development/testing only**. In production, use the On-Behalf-Of (OBO) flow via MSAL.

In [ ]:
# Acquire a user token for sensitivity label enforcement
# This uses your current Azure CLI login session
token_result = subprocess.run(
    ["az", "account", "get-access-token", "--resource", "https://search.azure.com", "--query", "accessToken", "--output", "tsv"],
    capture_output=True, text=True, check=True,
)
user_token = token_result.stdout.strip()
print(f"User token acquired (length: {len(user_token)} chars)")

In [ ]:
# Query with sensitivity label enforcement
# The SDK passes the user token via the x-ms-query-source-authorization header.
# Only documents the user is authorized to view (based on Purview label policies) are returned.
results = search_client.search(
    search_text="*",
    select=["chunk_id", "title", "content", "sensitivityLabel"],
    top=20,
    headers={
        "x-ms-query-source-authorization": user_token,  # User token — no "Bearer" prefix
    },
)

print("Search results with sensitivity label enforcement:\n")
doc_count = 0
for doc in results:
    doc_count += 1
    label = doc.get("sensitivityLabel", "N/A")
    content_preview = doc.get("content", "")[:100] + "..."
    print(f"  [{label}] {doc['title']}")
    print(f"    {content_preview}\n")

print(f"\nTotal documents returned: {doc_count}")
print("(Documents with labels the user cannot access are excluded from results)")

In [ ]:
# Try a targeted search — only results matching the user's label permissions are returned
results = search_client.search(
    search_text="financial projections revenue",
    select=["title", "content", "sensitivityLabel"],
    top=5,
    headers={
        "x-ms-query-source-authorization": user_token,
    },
)

print("Search for 'financial projections revenue' with label enforcement:\n")
doc_count = 0
for doc in results:
    doc_count += 1
    label = doc.get("sensitivityLabel", "N/A")
    content_preview = doc.get("content", "")[:150] + "..."
    print(f"  [{label}] {doc['title']}")
    print(f"    {content_preview}\n")

if doc_count == 0:
    print("  No results — the user may not have EXTRACT rights for the 'Highly Confidential' label.")
else:
    print(f"  {doc_count} result(s) returned.")

## Phase 3H: Elevated Read (Admin Investigation)

Elevated read lets an authorized admin return **all** labeled documents regardless of the user's label permissions. Every document returned generates an entry in the **Microsoft Purview audit log**.

Requirements:
- The calling user must hold the **Search Index Data Contributor** role (not just Reader)
- The `x-ms-enable-elevated-read` header is set to `true`
- **Cannot** be combined with `x-ms-query-source-authorization`

Use cases: compliance reviews, eDiscovery, incident response.

In [ ]:
# Elevated read — bypasses label filtering, creates Purview audit entries
# Requires Search Index Data Contributor role on the search service
results = search_client.search(
    search_text="*",
    select=["chunk_id", "title", "sensitivityLabel"],
    top=20,
    headers={
        "x-ms-enable-elevated-read": "true",
        # Note: x-ms-query-source-authorization is NOT allowed with elevated read
    },
)

print("Elevated read results (all documents, regardless of label permissions):\n")
label_counts = {}
for doc in results:
    label = doc.get("sensitivityLabel", "None")
    label_counts[label] = label_counts.get(label, 0) + 1
    print(f"  [{label:>30}]  {doc['title']}")

print(f"\nLabel distribution: {json.dumps(label_counts, indent=2)}")
print("\n⚠️  Each document returned has generated a Microsoft Purview audit log entry.")

## Phase 3I: Resolve Sensitivity Label Names (Optional)

The search index stores sensitivity labels as **GUIDs**. To display human-readable label names in your application, call the [Microsoft Graph Sensitivity Labels API](https://learn.microsoft.com/en-us/graph/api/sensitivitylabel-get) to resolve the GUID to a label name.

> **Requires**: `InformationProtectionPolicy.Read` permission on the calling application's Graph API registration. Skip this section if your app registration doesn't have this permission.

In [ ]:
import requests

# Get a Graph API token
graph_token_result = subprocess.run(
    ["az", "account", "get-access-token", "--resource", "https://graph.microsoft.com", "--query", "accessToken", "--output", "tsv"],
    capture_output=True, text=True, check=True,
)
graph_token = graph_token_result.stdout.strip()

# Fetch all sensitivity labels from the tenant
graph_headers = {"Authorization": f"Bearer {graph_token}"}
label_response = requests.get(
    "https://graph.microsoft.com/v1.0/security/informationProtection/sensitivityLabels",
    headers=graph_headers,
)

if label_response.status_code == 200:
    labels_data = label_response.json().get("value", [])
    label_lookup = {label["id"]: label["name"] for label in labels_data}
    print(f"Retrieved {len(label_lookup)} sensitivity labels from tenant:\n")
    for label_id, label_name in label_lookup.items():
        print(f"  {label_id} → {label_name}")
else:
    print(f"Failed to fetch labels (HTTP {label_response.status_code}): {label_response.text}")
    print("Ensure your app registration has InformationProtectionPolicy.Read permission.")
    label_lookup = {}

In [ ]:
# Display search results with resolved label names
if label_lookup:
    results = search_client.search(
        search_text="*",
        select=["title", "sensitivityLabel"],
        top=10,
        headers={"x-ms-enable-elevated-read": "true"},
    )

    print("Search results with resolved label names:\n")
    for doc in results:
        label_guid = doc.get("sensitivityLabel", "")
        label_name = label_lookup.get(label_guid, label_guid or "No label")
        print(f"  📄 {doc['title']}")
        print(f"     Label: {label_name} ({label_guid[:8]}...)\n")
else:
    print("Skipped — label lookup table is empty. Run the cell above first.")

## Phase 3J: Cleanup

Uncomment and run the cell below to delete the resources created in this notebook.

In [ ]:
# # Delete the indexer
# indexer_client.delete_indexer(INDEXER_NAME)
# print(f"Indexer '{INDEXER_NAME}' deleted.")

# # Delete the skillset
# indexer_client.delete_skillset(SKILLSET_NAME)
# print(f"Skillset '{SKILLSET_NAME}' deleted.")

# # Delete the data source connection
# indexer_client.delete_data_source_connection(DATA_SOURCE_NAME)
# print(f"Data source '{DATA_SOURCE_NAME}' deleted.")

# # Delete the index
# index_client.delete_index(INDEX_NAME)
# print(f"Index '{INDEX_NAME}' deleted.")

## Summary

In this notebook, you:

1. **Created a Purview-enabled index** with `purview_enabled=True` and a `sensitivityLabel` field marked with `sensitivity_label_id=True`
2. **Configured a data source** with `indexer_permission_options=["sensitivityLabel"]` to extract label metadata during indexing
3. **Built a skillset** with text splitting, Azure OpenAI embeddings, and index projections that propagate sensitivity labels from parent documents to child chunks
4. **Ran an indexer** that ingested labeled documents from Blob Storage and attached their Purview sensitivity labels
5. **Queried with label enforcement** using `x-ms-query-source-authorization` — only documents the user is authorized to access were returned
6. **Performed elevated read** for admin investigation with `x-ms-enable-elevated-read` — all documents returned regardless of label permissions, with Purview audit logging
7. *(Optional)* **Resolved label GUIDs** to human-readable names via the Microsoft Graph API

### Key Concepts

- **`purview_enabled`** is permanent — set once at index creation, cannot be changed later
- **Query-time enforcement** requires two tokens: application RBAC token (via SDK) + user identity token (via `x-ms-query-source-authorization`)
- **Elevated read** requires `Search Index Data Contributor` role and creates audit entries in Microsoft Purview
- **Index projections** are essential for integrated vectorization to propagate labels from parent documents to child chunks

### Learn More

- [Indexer setup for sensitivity labels](https://learn.microsoft.com/en-us/azure/search/search-indexer-sensitivity-labels)
- [Query-time enforcement](https://learn.microsoft.com/en-us/azure/search/search-query-sensitivity-labels)
- [Document-level security in Azure AI Search](https://learn.microsoft.com/en-us/azure/search/search-document-level-access-overview)
- [End-to-end reference sample](https://aka.ms/Ignite25/aisearch-purview-sensitivity-labels-repo)